# 🟢 PaddleOCR LOCAL FAST V2

**SOLO desarrollo local.**  
Este notebook se niega a instalar nada si no detecta `/content/work`,
para evitar ejecutar por accidente en Colab Cloud.

Versiones exactas:
- PaddlePaddle GPU **3.2.0**
- CUDA wheel **cu126**
- PaddleOCR **3.2.0**

No usa venv. No reinicia el kernel. El smoke test corre en un proceso Python nuevo.

In [ ]:
import sys, platform, pathlib, importlib.metadata as md

print("🟢 LOCAL FAST V2 · Paddle 3.2.0 / PaddleOCR 3.2.0")
print()

WORKSPACE = pathlib.Path("/content/work")
if not WORKSPACE.exists():
    raise RuntimeError(
        "ABORTADO: no detecto /content/work. "
        "Este notebook es SOLO para el Docker local. "
        "Conectá Colab a http://127.0.0.1:9000 y volvé a ejecutar."
    )

print("✅ Runtime local detectado:", WORKSPACE)
print("Python:", sys.version)
print("Platform:", platform.platform())

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"Esperaba Python 3.12 del Docker local; encontré {sys.version_info.major}.{sys.version_info.minor}."
    )

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return "NO INSTALADO"

print()
for name in ["paddlepaddle-gpu","paddleocr","paddlex","torch","pillow","numpy"]:
    print(f"{name:20} {ver(name)}")

In [ ]:
import sys, os, pathlib, subprocess, importlib.metadata as md

PADDLE = "3.2.0"
OCR = "3.2.0"
INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"

os.environ["PIP_CACHE_DIR"] = str(pathlib.Path.home()/".cache"/"pip")
os.environ["PADDLE_PDX_CACHE_HOME"] = str(pathlib.Path.home()/".cache"/"paddlex")
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

def run(cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

print("Pip cache persistente:", os.environ["PIP_CACHE_DIR"])
print("Model cache persistente:", os.environ["PADDLE_PDX_CACHE_HOME"])
print()

if ver("paddlepaddle-gpu") == PADDLE:
    print("✅ PaddlePaddle GPU 3.2.0 ya instalado.")
else:
    existing = ver("paddlepaddle-gpu")
    if existing:
        raise RuntimeError(
            f"Hay PaddlePaddle GPU {existing} instalado. "
            "No voy a modificarlo automáticamente. Usá reset local si querés volver a limpio."
        )
    print("⬇️ Primera instalación de Paddle 3.2.0.")
    print("Este wheel es grande; esta descarga debería ocurrir UNA sola vez.")
    run([
        sys.executable, "-m", "pip", "install",
        "--retries", "10",
        "--timeout", "120",
        f"paddlepaddle-gpu=={PADDLE}",
        "-i", INDEX,
    ])

if ver("paddleocr") == OCR:
    print("✅ PaddleOCR 3.2.0 ya instalado.")
else:
    existing = ver("paddleocr")
    if existing:
        raise RuntimeError(
            f"Hay PaddleOCR {existing} instalado. "
            "No voy a mezclar versiones automáticamente."
        )
    run([
        sys.executable, "-m", "pip", "install",
        "--retries", "10",
        "--timeout", "120",
        f"paddleocr=={OCR}",
    ])

print()
print("✅ Instalación lista.")
print("NO reinicies el kernel: la siguiente celda usa un proceso Python nuevo.")

In [ ]:
import sys, subprocess, pathlib, os

worker_path = pathlib.Path("/content/work/.colab-dev/paddle_smoke_worker_v2.py")
worker_path.parent.mkdir(parents=True, exist_ok=True)
worker_path.write_text('\nimport os, sys, json, time\nfrom pathlib import Path\n\nos.environ.setdefault("PADDLE_PDX_MODEL_SOURCE", "BOS")\nos.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(Path.home()/".cache"/"paddlex"))\n\nprint("=== WORKER NUEVO ===", flush=True)\nprint("Python:", sys.version, flush=True)\n\nimport paddle\nprint("Paddle:", paddle.__version__, flush=True)\nprint("CUDA:", paddle.is_compiled_with_cuda(), flush=True)\nprint("GPU count:", paddle.device.cuda.device_count(), flush=True)\n\nif not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() < 1:\n    raise RuntimeError("Paddle no ve la GPU CUDA.")\n\npaddle.set_device("gpu:0")\n\nimport PIL\nfrom PIL import Image, ImageDraw\nprint("Pillow:", PIL.__version__, flush=True)\n\nfrom paddleocr import PaddleOCR\nimport paddleocr\nprint("PaddleOCR:", getattr(paddleocr, "__version__", "unknown"), flush=True)\n\nimg_path = Path("/content/work/.colab-dev/paddle_smoke_es.png")\nimg_path.parent.mkdir(parents=True, exist_ok=True)\nimg = Image.new("RGB", (1400, 320), "white")\nImageDraw.Draw(img).text(\n    (50, 100),\n    "Histologia epitelio plano simple prueba OCR espanol 12345",\n    fill="black"\n)\nimg.save(img_path)\n\nprint("Inicializando OCR...", flush=True)\nocr = PaddleOCR(\n    lang="es",\n    device="gpu:0",\n    use_doc_orientation_classify=False,\n    use_doc_unwarping=False,\n    use_textline_orientation=False,\n)\n\nprint("Ejecutando OCR...", flush=True)\nresults = ocr.predict(str(img_path))\ntexts = []\nfor res in results:\n    d = getattr(res, "json", res)\n    if callable(d):\n        d = d()\n    if isinstance(d, dict) and "res" in d:\n        d = d["res"]\n    if isinstance(d, dict):\n        texts += [str(x) for x in d.get("rec_texts", [])]\n\nprint("Textos:", json.dumps(texts, ensure_ascii=False), flush=True)\nif not texts:\n    raise RuntimeError("No se reconoció texto.")\n\nprint("✅ SMOKE TEST COMPLETO", flush=True)\n', encoding="utf-8")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PADDLE_PDX_MODEL_SOURCE"] = "BOS"
env["PADDLE_PDX_CACHE_HOME"] = str(pathlib.Path.home()/".cache"/"paddlex")

print("Ejecutando worker fresco...")
proc = subprocess.Popen(
    [sys.executable, str(worker_path)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc:
    raise RuntimeError(f"Worker falló con código {rc}")